# 自定义中间件-Wrap-style-hooks
## 1、wrap_model_call的使用
### 1.1基于装饰器的实现

In [4]:
import os
from typing import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [2]:
from langchain.agents.middleware import wrap_model_call,ModelResponse,ModelRequest

@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse | None:
        request.messages[-1].content +="--->wrap_model_call_middleware_before <---"
        #模型的调用
        response =  handler(request)
        response.result[0].content +="--->wrap_model_call_after <---"
        return response


In [7]:
agent = create_agent(
     model=model,
     middleware=[wrap_model_call_middleware],
)

response = agent.invoke({
     "messages":[HumanMessage("你好牛逼")]
 })

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好牛逼--->wrap_model_call_middleware_before <---
================================== Ai Message ==================================

看起来你发了一段有点特别的文本！让我帮你解读一下：

**“你好牛逼”** — 这是一句中文口语，意思是“你很厉害/很棒”，通常是夸奖，但语气偏随意、网络化。

**`--->wrap_model_call_middleware_before <---`** — 这看起来像是某种技术标记或钩子（hook）名称，常见于：

- **LangChain / LangGraph** 中的中间件（middleware）命名风格
- **Agent 框架**里包裹模型调用的钩子，`before` 表示在模型调用**之前**执行
- 类似 `wrap_model_call_middleware_before` 这种命名，通常是用来在 LLM 请求发出前做拦截、改写、注入上下文等操作

所以整体看，你可能是在：
1. 测试某个 Agent/LLM 框架的中间件触发点
2. 或者只是随手丢了一段“夸奖 + 技术标记”的组合，看我会怎么反应 😄

如果你是想问关于 `wrap_model_call_middleware_before` 这个中间件的具体用法或实现，告诉我你用的是哪个框架（LangChain、LangGraph、还是自研的），我可以帮你详细讲讲怎么写、什么时候触发、能做什么。

如果只是路过打个招呼——那也谢谢你，我收下这句夸奖了 🙌--->wrap_model_call_after <---
